# Train the narration MLP head
Run after dataset creation. This notebook never loads Magi and uses only cached embeddings and labels.
Select thresholds and checkpoints by narration F1 on the natural validation distribution.
Architecture: frozen DETR token → Linear(feature_dim, 64) → ReLU → Linear(64, 1).
This experiment tests a nonlinear classifier on frozen features.


In [1]:
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

ROOT = Path(r'C:\Users\ESHAAN\BYTE\Applied-Task\Magi\isNarratorTextMagi')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
BATCH_SIZE = 32
HIDDEN_DIM = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 30
PATIENCE = 5
MIN_F1_IMPROVEMENT = 1e-4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

data = torch.load(ROOT / 'narration_data.pt', map_location='cpu', weights_only=True)
assert data['settings']['data_source'] == 'development'
assert data['settings'].get('split_strategy') == 'surviving_tokens_narration_and_sequence_v2', 'Regenerate the dataset with the updated split notebook.'
samples = data['samples']
FEATURE_DIM = samples[0]['detr_embedding'].shape[-1]
assert FEATURE_DIM == data['feature_dim']
assert all(s['detr_embedding'].shape == (FEATURE_DIM,) for s in samples)
assert all(torch.isfinite(s['detr_embedding']).all() for s in samples)
train_indices = data['train_sample_indices']
validation_indices = data['splits']['validation']
assert set(train_indices) <= set(data['splits']['train'])
assert set(train_indices).isdisjoint(validation_indices)
assert set(data['sequence_splits']['train']).isdisjoint(data['sequence_splits']['validation'])
assert {samples[i]['page_path'] for i in train_indices}.isdisjoint({samples[i]['page_path'] for i in validation_indices})
assert {samples[i]['page_sha256'] for i in train_indices}.isdisjoint({samples[i]['page_sha256'] for i in validation_indices})
assert all({samples[i]['label'] for i in indices} == {0, 1} for indices in [train_indices, validation_indices])
print('Train:', len(train_indices), '| Validation:', len(validation_indices), '| Device:', DEVICE)

print('Narration audit:', data['narration_audit'])
print('Narration support | Train:', sum(samples[i]['label'] for i in train_indices),
      '| Validation:', sum(samples[i]['label'] for i in validation_indices))


Train: 150 | Validation: 297 | Device: cuda
Narration audit: {'gt': 76, 'accepted_labels': 63, 'matched_tokens': 63}
Narration support | Train: 50 | Validation: 13


In [2]:
class NarrationHead(nn.Module):
    def __init__(self, feature_dim, hidden_dim):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.classifier(x).squeeze(-1)


class NarrationEmbeddingDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return sample['detr_embedding'].float(), torch.tensor(sample['label'], dtype=torch.float32)


train_dataset = NarrationEmbeddingDataset([samples[i] for i in train_indices])
validation_dataset = NarrationEmbeddingDataset([samples[i] for i in validation_indices])
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
validation_loader = DataLoader(validation_dataset, batch_size=64, shuffle=False)
head = NarrationHead(FEATURE_DIM, HIDDEN_DIM).to(DEVICE)
assert set(dict(head.named_parameters())) == {
    'classifier.0.weight', 'classifier.0.bias', 'classifier.2.weight', 'classifier.2.bias'}
trainable_count = sum(p.numel() for p in head.parameters() if p.requires_grad)
assert trainable_count == (FEATURE_DIM + 1) * HIDDEN_DIM + HIDDEN_DIM + 1
print('Trainable parameters:', trainable_count)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


Trainable parameters: 16513


## Metrics and midpoint threshold selection

In [3]:
def classification_metrics(labels, probabilities, threshold):
    labels = np.asarray(labels, dtype=np.int64)
    predictions = np.asarray(probabilities, dtype=np.float64) >= threshold
    tp = int(((labels == 1) & predictions).sum())
    fp = int(((labels == 0) & predictions).sum())
    tn = int(((labels == 0) & ~predictions).sum())
    fn = int(((labels == 1) & ~predictions).sum())
    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else 0.0
    negative_recall = tn / (tn + fp) if tn + fp else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    f0_5 = 1.25 * tp / (1.25 * tp + fp + .25 * fn) if 1.25 * tp + fp + .25 * fn else 0.0
    return dict(tp=tp, fp=fp, tn=tn, fn=fn, narration_precision=precision,
        narration_recall=recall, narration_f1=f1, narration_f0_5=f0_5,
        non_narration_recall=negative_recall, balanced_accuracy=(recall + negative_recall) / 2,
        accuracy=(tp + tn) / len(labels), predicted_narration=tp + fp, actual_narration=tp + fn)


def selection_key(metrics):
    precision = metrics['narration_precision']
    return (metrics['narration_f1'], precision if precision is not None else -1.0,
            metrics['narration_recall'], metrics['balanced_accuracy'])


def select_threshold(labels, probabilities):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    assert np.isfinite(probabilities).all()
    values = np.unique(probabilities)
    midpoints = values[:-1] + (values[1:] - values[:-1]) / 2
    thresholds = np.concatenate(([0.0], midpoints, [np.nextafter(1.0, np.inf)]))
    best_threshold = None
    best_metrics = None
    for threshold in thresholds:
        metrics = classification_metrics(labels, probabilities, float(threshold))
        if best_metrics is None or selection_key(metrics) > selection_key(best_metrics):
            best_threshold = float(threshold)
            best_metrics = metrics
    return best_threshold, best_metrics


@torch.no_grad()
def evaluate_head():
    head.eval()
    loss_total = 0.0
    labels_all = []
    probabilities_all = []
    for features, labels in validation_loader:
        features, labels = features.to(DEVICE), labels.to(DEVICE)
        logits = head(features)
        loss_total += loss_fn(logits, labels).item() * len(labels)
        probabilities_all.extend(torch.sigmoid(logits).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())
    threshold, metrics = select_threshold(labels_all, probabilities_all)
    return loss_total / len(validation_dataset), threshold, metrics


## Train, save the best F1 checkpoint, and stop after five stale epochs

In [4]:
best_metrics = None
patience_best_f1 = -1.0
stale_epochs = 0
history = []
for epoch in range(1, EPOCHS + 1):
    head.train()
    loss_total = 0.0
    for features, labels in train_loader:
        features, labels = features.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = head(features)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        loss_total += loss.item() * len(labels)
    train_loss = loss_total / len(train_dataset)
    validation_loss, threshold, metrics = evaluate_head()
    history.append(dict(epoch=epoch, train_loss=train_loss, validation_loss=validation_loss,
                        threshold=threshold, **metrics))
    precision = metrics['narration_precision']
    precision_text = f'{precision:.3f}' if precision is not None else 'undefined'
    print(f"Epoch {epoch:02d} | train {train_loss:.3f} | val {validation_loss:.3f} | "
          f"P {precision_text} | R {metrics['narration_recall']:.3f} | "
          f"F1 {metrics['narration_f1']:.3f} | F0.5 {metrics['narration_f0_5']:.3f} | "
          f"NR {metrics['non_narration_recall']:.3f} | BA {metrics['balanced_accuracy']:.3f} | "
          f"Acc {metrics['accuracy']:.3f} | threshold {threshold:.6f} | "
          f"TP/FP/TN/FN {metrics['tp']}/{metrics['fp']}/{metrics['tn']}/{metrics['fn']} | "
          f"pred/actual {metrics['predicted_narration']}/{metrics['actual_narration']}")
    if best_metrics is None or selection_key(metrics) > selection_key(best_metrics):
        best_metrics = dict(metrics)
        checkpoint = dict(state_dict={k: v.detach().cpu().clone() for k, v in head.state_dict().items()},
            feature_dim=FEATURE_DIM, architecture={'type': 'mlp', 'hidden_dim': HIDDEN_DIM, 'activation': 'relu'},
            validation_threshold=threshold, validation=dict(metrics),
            validation_loss=validation_loss, best_epoch=epoch, selection_metric='narration_f1',
            settings=dict(seed=SEED, negative_ratio=data['settings']['negative_ratio'],
                learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE,
                epochs=EPOCHS, patience=PATIENCE, min_f1_improvement=MIN_F1_IMPROVEMENT),
            magi_path=data['settings']['magi_path'], model_identity=data['model_identity'],
            dataset_id=data['dataset_id'])
        torch.save(checkpoint, ROOT / 'narration_head.pt')
    if metrics['narration_f1'] > patience_best_f1 + MIN_F1_IMPROVEMENT:
        patience_best_f1 = metrics['narration_f1']
        stale_epochs = 0
    else:
        stale_epochs += 1
    if stale_epochs >= PATIENCE:
        print('Early stopping.')
        break
print('Saved best epoch:', checkpoint['best_epoch'], '| F1:', round(best_metrics['narration_f1'], 4))
print('Checkpoint:', ROOT / 'narration_head.pt')


Epoch 01 | train 0.642 | val 0.427 | P 0.200 | R 0.385 | F1 0.263 | F0.5 0.221 | NR 0.930 | BA 0.657 | Acc 0.906 | threshold 0.402283 | TP/FP/TN/FN 5/20/264/8 | pred/actual 25/13
Epoch 02 | train 0.591 | val 0.306 | P 0.238 | R 0.385 | F1 0.294 | F0.5 0.258 | NR 0.944 | BA 0.664 | Acc 0.919 | threshold 0.320082 | TP/FP/TN/FN 5/16/268/8 | pred/actual 21/13
Epoch 03 | train 0.585 | val 0.323 | P 0.200 | R 0.615 | F1 0.302 | F0.5 0.231 | NR 0.887 | BA 0.751 | Acc 0.875 | threshold 0.339388 | TP/FP/TN/FN 8/32/252/5 | pred/actual 40/13
Epoch 04 | train 0.542 | val 0.381 | P 0.250 | R 0.462 | F1 0.324 | F0.5 0.275 | NR 0.937 | BA 0.699 | Acc 0.916 | threshold 0.452255 | TP/FP/TN/FN 6/18/266/7 | pred/actual 24/13
Epoch 05 | train 0.523 | val 0.405 | P 0.235 | R 0.615 | F1 0.340 | F0.5 0.268 | NR 0.908 | BA 0.762 | Acc 0.896 | threshold 0.484248 | TP/FP/TN/FN 8/26/258/5 | pred/actual 34/13
Epoch 06 | train 0.501 | val 0.365 | P 0.276 | R 0.615 | F1 0.381 | F0.5 0.310 | NR 0.926 | BA 0.771 | Ac